# A5 — signature reversal and nearest measured cell lines

Replaces the ODE panel. Every displayed viability is a measurement.
Smoke/CI uses `synthetic_smoke` or a committed table proxy — never silent full-LINCS labelling.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = False

def cohort_ids():
    import pandas as pd
    assign = V3 / "cluster_assignments.parquet"
    if assign.is_file():
        return pd.read_parquet(assign)["patient_id"].astype(str).str[:12].unique().tolist()
    expr = INTERIM / "intrinsic_expression.parquet"
    if expr.is_file():
        return pd.read_parquet(expr).index.astype(str).str[:12].unique().tolist()
    return None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        ids = cohort_ids()
        if ids is not None:
            kwargs["sample_ids"] = ids
            kwargs.setdefault("n", len(ids))
            kwargs.setdefault("cohort", True)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
import pandas as pd
from gctx_retrieval import load_perturbations, SOURCE_SMOKE
from v3_real import persist_real

paths = {
    "compact_matrix": REPO_ROOT / "outputs" / "copilot_artifacts" / "compact_gctx.parquet",
    "committed_table": REPO_ROOT / "results" / "mofa_clusters" / "slide_drug_retrieval_table.csv",
}
mat, _meta, source = load_perturbations(paths)
print("reversal source", source)

cohort_path = V3 / "cohort_payload.json"
if not cohort_path.is_file() or json.loads(cohort_path.read_text()).get("synthetic_samples", 0):
    print("A5 assembling real cohort — never using assemble_v3")
    persist_real(V2_ROOT, REPO_ROOT, n_boot=8 if SMOKE_TEST else 50, n_init=3 if SMOKE_TEST else 10)
cohort = json.loads(cohort_path.read_text())
if int(cohort.get("synthetic_samples") or 0) > 0:
    raise RuntimeError("A5 refused a synthetic cohort_payload")
patients = {p.stem.replace("payload_", ""): json.loads(p.read_text()) for p in cohort_path.parent.glob("payload_*.json")}

source_note = source if not mat.empty else "unavailable"
if source_note == SOURCE_SMOKE:
    source_note = "unavailable"
    print("A5: perturbation matrix is smoke/proxy — reversal not scored")

line_rows, curve_rows = [], []
for pid, payload in patients.items():
    for line in payload.get("nearest_lines") or []:
        line_rows.append({"patient_id": pid, **{k: v for k, v in line.items() if k != "curves"}})
        for curve in line.get("curves") or []:
            curve_rows.append({"patient_id": pid, "line_id": line["line_id"], **{k: v for k, v in curve.items() if k not in {"concentration_nm", "viability", "lower", "upper"}},
                               "points": json.dumps({k: curve[k] for k in ("concentration_nm", "viability", "lower", "upper")})})
pd.DataFrame(line_rows).to_parquet(V3 / "nearest_cell_lines.parquet")
pd.DataFrame(curve_rows).to_parquet(V3 / "dose_response_curves.parquet")
a5 = cohort.get("gates", {}).get("a5") or {}
conc = a5.get("nearest_line_subtype_concordance") or {"concordance": 0, "chance": None, "n": 0}
pos = a5.get("known_drug_positive_control") or {"hits": [], "passed": False}
(V3 / "a5_meta.json").write_text(json.dumps({"source": source_note, "positive_control": pos, "concordance": conc}, indent=2))
print(source_note, pos, conc)


In [ ]:
meta = json.loads((V3 / "a5_meta.json").read_text())
pos = meta["positive_control"]
gate("NB_A5", "known_drug_positive_control", float(len(pos.get("hits") or [])), 1,
     note=f"ER cluster endocrine hits: {pos.get('hits')} source={meta['source']}")
conc = meta["concordance"]
gate("NB_A5", "nearest_line_subtype_concordance", float(conc.get("concordance") or 0), 0.40,
     note=f"chance={conc.get('chance')} n={conc.get('n')}")
